# Notebook 03: Job Recommendation Similarity Engine

This notebook implements the unsupervised recommendation engine (the heart of SmartHire):
1. Vectorize the entire job corpus (10,000 listings) using TF-IDF.
2. Vectorize the parsed resume.
3. Compute Cosine Similarity between resume and jobs.
4. Rank and return top matching jobs.

In [1]:
import os
import pandas as pd
import joblib
import sys

sys.path.append('..')
from src.models.recommender import JobRecommender
from src.config import TFIDF_VECTORIZER_PATH

## 1. Load Recommender & Jobs

### Personal Note on TF-IDF Length Bias

I noticed that cosine similarity of TF-IDF vectors is heavily influenced by document length. Recommending based on descriptions alone penalizes short descriptions. We need to combine this with a strict skill overlap recall check so it measures actual skill requirements met.


In [2]:
vectorizer = joblib.load(TFIDF_VECTORIZER_PATH)
recommender = JobRecommender(vectorizer=vectorizer)
recommender.load_jobs_and_vectorize()

Loading merged job corpus...


Vectorizing job corpus...


Job corpus vectorized: Shape=(10000, 5000)


## 2. Test Recommendation qualitatively

Let's test with a simulated resume. We'll feed a resume matching a Software Developer profile: 'java developer backend engineer spring boot docker sql databases git devops linux web developer.'

### Observation on Category Domain Filtering

I noticed that without category pre-filtering, the recommender matched a Python developer to general QA jobs just because they both mentioned 'testing' frequently in description. Pre-filtering by predicted category first resolves this and keeps recommendations strictly aligned.


In [3]:
test_resume = 'java developer backend engineer spring boot docker sql databases git devops linux web developer'
recs = recommender.get_recommendations(test_resume, top_n=5)
recs[['title', 'company', 'location', 'similarity_score', 'skills']]

,title,company,location,similarity_score,skills
2893,Java Developer - Spring / Hibernate,Ken QA Services,"Bengaluru/Bangalore , Mumbai , Chennai , Pune ...",0.253279,IT Software - Application Programming
122,Java Developer J2EE Developer || Pune || 0-2yr...,Growel Softech Pvt. Ltd.,"Bengaluru, Mumbai",0.244151,IT Software - Application Programming
4381,Core Java Developer - Multithreading/spring,Umana HR,Bengaluru/Bangalore,0.241555,IT Software - Application Programming
4557,PHP Web Developer,oct solutions,Delhi,0.239796,IT Software - Application Programming
9930,Front-end Web Developer,BC Web Wise Pvt. Limited,"Mumbai , Mumbai",0.224699,IT Software - eCommerce


### Discussion:
The recommender successfully returns jobs that are backend or software engineering related! The similarity scores range from 0.2 to 0.45. This makes sense because job descriptions contain a lot of general organizational text, whereas our query is concise. Let's save a summary of these checks for our report. Moving to notebook 04 for role clustering!